# Behaviour Dataset Processing

## Overview

This notebook processes a single National Travel Survey (NTS) behavioural dataset.

The objective is to clean, harmonise, and transform mobility behaviour information into a standardised behavioural representation for the proposed representative population framework.

The processed behavioural data will be exported as a standardised JSON file for subsequent integration with the demographic layer.

## Dataset Information

| Item | Description |
|------|-------------|
| Dataset |NTS0502: Trip start time by trip purpose (Monday to Friday only): England, 2002 onwards|
| Source |National Travel Survey|
| Geography | England |
| Input File |nts0502.ods
| Output File |nts0502b_pattern_temporal_probability.json|
| Notebook Author | Peter |
| Last Updated |2026-07-14|

## 1. Import

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

## 2. Configuration

In [2]:
from pathlib import Path

# File paths

input_file = "nts0502.ods"
output_file = "nts0502b_temporal_pattern_probability.json"

raw_data_path = Path("../../data/nts/raw")
output_data_path = Path("../../data/nts/harmonised")

## 3. Load Dataset

In [3]:
df = pd.read_excel(
    raw_data_path / input_file,
    sheet_name="NTS0502b_purpose_by_start_time",
    skiprows=5
)

df_processed = df.copy()

df_processed.head()

,Year [note 2],Trip purpose,0000 to 0059 (%),0100 to 0159 (%),0200 to 0259 (%),0300 to 0359 (%),0400 to 0459 (%),0500 to 0559 (%),0600 to 0659 (%),0700 to 0759 (%),0800 to 0859 (%),0900 to 0959 (%),1000 to 1059 (%),1100 to 1159 (%),1200 to 1259 (%),1300 to 1359 (%),1400 to 1459 (%),1500 to 1559 (%),1600 to 1659 (%),1700 to 1759 (%),1800 to 1859 (%),1900 to 1959 (%),2000 to 2059 (%),2100 to 2159 (%),2200 to 2259 (%),2300 to 2359 (%),All day (%),Unweighted sample size: trips (thousands) (number)
0,2002 to 2006,Commuting,0.205698,0.101931,0.104566,0.182574,0.611806,2.449245,6.286387,15.986476,15.087049,3.656802,1.434472,1.576268,2.830811,3.998375,3.266973,4.087944,9.286012,15.321603,6.496979,2.288845,1.582217,1.308415,1.182520,0.666031,100,207.075
1,2002 to 2006,Business,0.113469,0.076103,0.047455,0.036680,0.139022,0.742937,2.643129,7.036194,10.194576,7.817521,6.798939,7.197912,7.581983,7.629309,6.934835,7.728231,9.433110,7.337461,4.259686,2.396614,1.487375,1.135503,0.802173,0.429784,100,48.075
2,2002 to 2006,Education,0.004818,0.000000,0.001001,0.003418,0.002020,0.006047,0.082235,6.011576,41.064773,2.343561,1.312823,1.452119,2.073874,1.840750,2.279897,32.743377,4.877663,2.249846,0.765730,0.279634,0.289271,0.227910,0.058754,0.028904,100,99.820
3,2002 to 2006,Escort education,0.001225,0.000000,0.004095,0.028255,0.002498,0.005781,0.045425,2.077776,35.409744,7.974474,0.520823,2.095834,1.765297,0.923104,7.652706,35.109558,3.713478,1.677470,0.426319,0.181188,0.221839,0.130278,0.025667,0.007166,100,75.727
4,2002 to 2006,Shopping,0.023158,0.003951,0.000420,0.003484,0.005803,0.043154,0.201715,0.966488,2.895945,8.350572,12.269701,12.792924,10.395929,9.251785,9.247824,8.733509,7.756958,5.720232,4.792887,3.372280,1.852103,0.890751,0.307148,0.121280,100,202.265


## 4. Dataset Exploration

Inspect the structure of the dataset before processing.

In [4]:
df_processed.info()

print(f"Shape: {df_processed.shape}")

df_processed.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 162 entries, 0 to 161
Data columns (total 28 columns):
 #   Column                                              Non-Null Count  Dtype  
---  ------                                              --------------  -----  
 0   Year [note 2]                                       162 non-null    object 
 1   Trip purpose                                        162 non-null    object 
 2   0000 to 0059 (%)                                    162 non-null    float64
 3   0100 to 0159 (%)                                    162 non-null    float64
 4   0200 to 0259 (%)                                    162 non-null    float64
 5   0300 to 0359 (%)                                    162 non-null    float64
 6   0400 to 0459 (%)                                    162 non-null    float64
 7   0500 to 0559 (%)                                    162 non-null    float64
 8   0600 to 0659 (%)                                    162 non-null    float64
 9  

Index(['Year [note 2]', 'Trip purpose', '0000 to 0059 (%)', '0100 to 0159 (%)',
       '0200 to 0259 (%)', '0300 to 0359 (%)', '0400 to 0459 (%)',
       '0500 to 0559 (%)', '0600 to 0659 (%)', '0700 to 0759 (%)',
       '0800 to 0859 (%)', '0900 to 0959 (%)', '1000 to 1059 (%)',
       '1100 to 1159 (%)', '1200 to 1259 (%)', '1300 to 1359 (%)',
       '1400 to 1459 (%)', '1500 to 1559 (%)', '1600 to 1659 (%)',
       '1700 to 1759 (%)', '1800 to 1859 (%)', '1900 to 1959 (%)',
       '2000 to 2059 (%)', '2100 to 2159 (%)', '2200 to 2259 (%)',
       '2300 to 2359 (%)', 'All day (%)',
       'Unweighted sample size: trips (thousands) (number)'],
      dtype='object')

## 5. Data Cleaning

Check the dataset quality before feature processing.

In [5]:
# Missing values
df_processed.isna().sum()

Year [note 2]                                         0
Trip purpose                                          0
0000 to 0059 (%)                                      0
0100 to 0159 (%)                                      0
0200 to 0259 (%)                                      0
0300 to 0359 (%)                                      0
0400 to 0459 (%)                                      0
0500 to 0559 (%)                                      0
0600 to 0659 (%)                                      0
0700 to 0759 (%)                                      0
0800 to 0859 (%)                                      0
0900 to 0959 (%)                                      0
1000 to 1059 (%)                                      0
1100 to 1159 (%)                                      0
1200 to 1259 (%)                                      0
1300 to 1359 (%)                                      0
1400 to 1459 (%)                                      0
1500 to 1559 (%)                                

In [6]:
# Duplicate records
df_processed.duplicated().sum()

np.int64(0)

In [7]:
# Summary statistics
df_processed.describe(include="all")

,Year [note 2],Trip purpose,0000 to 0059 (%),0100 to 0159 (%),0200 to 0259 (%),0300 to 0359 (%),0400 to 0459 (%),0500 to 0559 (%),0600 to 0659 (%),0700 to 0759 (%),0800 to 0859 (%),0900 to 0959 (%),1000 to 1059 (%),1100 to 1159 (%),1200 to 1259 (%),1300 to 1359 (%),1400 to 1459 (%),1500 to 1559 (%),1600 to 1659 (%),1700 to 1759 (%),1800 to 1859 (%),1900 to 1959 (%),2000 to 2059 (%),2100 to 2159 (%),2200 to 2259 (%),2300 to 2359 (%),All day (%),Unweighted sample size: trips (thousands) (number)
count,162,162,162.000000,162.000000,162.000000,162.000000,162.000000,162.000000,162.000000,162.000000,162.000000,162.000000,162.000000,162.000000,162.000000,162.000000,162.000000,162.000000,162.000000,162.000000,162.000000,162.000000,162.000000,162.000000,162.000000,162.000000,162.0,162.000000
unique,18,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,2002 to 2006,Commuting,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,9,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,0.089058,0.043278,0.036894,0.049394,0.146507,0.618636,1.818267,5.654573,14.710233,5.927349,5.602100,5.867082,5.638097,5.313716,6.482007,13.468072,7.572187,7.181043,5.069736,3.460791,2.155517,1.477477,1.040248,0.577738,100.0,194.683506
std,NaN,NaN,0.087903,0.046262,0.039748,0.057166,0.202371,0.873051,2.177353,4.382635,13.194212,2.111757,3.764071,3.694533,2.968122,2.750093,2.146009,10.491174,1.671384,3.557068,2.904026,2.652998,1.708746,1.391622,1.102508,0.700189,0.0,283.726031
min,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.040046,0.778843,1.975345,1.359084,0.205464,0.439194,0.568263,0.519388,2.279897,4.001963,3.264173,1.552432,0.078029,0.021115,0.037201,0.000000,0.000000,0.000000,100.0,1.754000
25%,NaN,NaN,0.010990,0.005429,0.002480,0.006510,0.006966,0.030323,0.261140,2.582247,4.852602,4.099684,1.592366,1.667619,2.224648,3.115778,5.772123,7.467507,7.016570,5.711452,4.170299,2.226134,1.443178,0.715605,0.285875,0.092938,100.0,49.662250
50%,NaN,NaN,0.058814,0.028334,0.030558,0.032145,0.047815,0.230295,0.804104,4.461597,10.294179,6.437383,5.983406,6.302384,6.286213,5.984169,6.621043,8.350193,7.846699,7.506322,5.627897,3.544970,1.903680,1.270236,0.819912,0.432535,100.0,93.948500
75%,NaN,NaN,0.136310,0.061583,0.050910,0.063077,0.174903,0.710326,2.287616,7.339498,14.847882,7.764267,8.118213,7.896013,7.490164,7.164542,7.922310,11.317629,8.750235,8.480188,6.695808,4.180962,2.608289,1.749069,1.293912,0.694793,100.0,197.937250


## 6. Feature Processing

Process the behavioural information required for the representative population framework.

The processing workflow in this section depends on the specific NTS dataset.

In [8]:
# Select latest period
df_processed = df_processed[
    df_processed["Year [note 2]"] == "2023 to 2024"
].copy()

df_processed = df_processed.rename(columns={
    "Year [note 2]": "Year"
})
# Columns to keep
base_columns = ["Year", "Trip purpose"]

hourly_time_columns = [
    "0000 to 0059 (%)",
    "0100 to 0159 (%)",
    "0200 to 0259 (%)",
    "0300 to 0359 (%)",
    "0400 to 0459 (%)",
    "0500 to 0559 (%)",
    "0600 to 0659 (%)",
    "0700 to 0759 (%)",
    "0800 to 0859 (%)",
    "0900 to 0959 (%)",
    "1000 to 1059 (%)",
    "1100 to 1159 (%)",
    "1200 to 1259 (%)",
    "1300 to 1359 (%)",
    "1400 to 1459 (%)",
    "1500 to 1559 (%)",
    "1600 to 1659 (%)",
    "1700 to 1759 (%)",
    "1800 to 1859 (%)",
    "1900 to 1959 (%)",
    "2000 to 2059 (%)",
    "2100 to 2159 (%)",
    "2200 to 2259 (%)",
    "2300 to 2359 (%)",
    "All day (%)"
]

df_processed = df_processed[base_columns + hourly_time_columns]

In [9]:
df_processed.head()

,Year,Trip purpose,0000 to 0059 (%),0100 to 0159 (%),0200 to 0259 (%),0300 to 0359 (%),0400 to 0459 (%),0500 to 0559 (%),0600 to 0659 (%),0700 to 0759 (%),0800 to 0859 (%),0900 to 0959 (%),1000 to 1059 (%),1100 to 1159 (%),1200 to 1259 (%),1300 to 1359 (%),1400 to 1459 (%),1500 to 1559 (%),1600 to 1659 (%),1700 to 1759 (%),1800 to 1859 (%),1900 to 1959 (%),2000 to 2059 (%),2100 to 2159 (%),2200 to 2259 (%),2300 to 2359 (%),All day (%)
153,2023 to 2024,Commuting,0.239840,0.148222,0.111912,0.218639,0.827896,3.058740,7.965453,15.264471,13.328675,3.658019,1.825335,1.631218,2.191046,3.136569,3.486577,5.383947,10.382031,13.261369,6.275847,2.542718,1.758006,1.131909,1.399447,0.772115,100
154,2023 to 2024,Business,0.041416,0.015177,0.060895,0.081876,0.220996,1.074968,4.589156,8.696725,11.267973,7.465897,6.009973,7.555960,6.552888,6.630029,6.343139,7.162665,9.123441,7.463486,3.991702,1.922051,1.518446,1.129986,0.698915,0.382243,100
155,2023 to 2024,Education,0.000000,0.000000,0.000000,0.030642,0.004397,0.011165,0.253959,9.799807,37.755919,1.758576,0.968553,1.160566,1.471589,0.844463,2.796316,33.281601,5.483792,2.674043,0.772700,0.471058,0.207967,0.186490,0.049809,0.016588,100
156,2023 to 2024,Escort education,0.000000,0.000000,0.007056,0.012924,0.004907,0.022035,0.181196,5.149004,38.714753,3.408739,0.375344,1.210844,1.091111,0.519388,9.301491,30.963633,5.211611,2.884029,0.528071,0.260502,0.096282,0.020906,0.030402,0.005771,100
157,2023 to 2024,Shopping,0.016551,0.002386,0.000000,0.000000,0.001099,0.047941,0.335906,1.288284,3.181589,7.245089,11.792016,12.445005,11.116141,9.373412,8.713807,7.572346,7.269566,6.474185,5.717458,4.203831,2.133739,0.761697,0.220305,0.087645,100


In [10]:
temporal_probability_matrix = df_processed.copy()

temporal_probability_matrix[hourly_time_columns] = (
    temporal_probability_matrix[hourly_time_columns] / 100
)

# Remove the all-day probability column

temporal_probability_matrix = temporal_probability_matrix.drop(
    columns=["All day (%)"]
)

display(temporal_probability_matrix)

,Year,Trip purpose,0000 to 0059 (%),0100 to 0159 (%),0200 to 0259 (%),0300 to 0359 (%),0400 to 0459 (%),0500 to 0559 (%),0600 to 0659 (%),0700 to 0759 (%),0800 to 0859 (%),0900 to 0959 (%),1000 to 1059 (%),1100 to 1159 (%),1200 to 1259 (%),1300 to 1359 (%),1400 to 1459 (%),1500 to 1559 (%),1600 to 1659 (%),1700 to 1759 (%),1800 to 1859 (%),1900 to 1959 (%),2000 to 2059 (%),2100 to 2159 (%),2200 to 2259 (%),2300 to 2359 (%)
153,2023 to 2024,Commuting,0.002398,0.001482,0.001119,0.002186,0.008279,0.030587,0.079655,0.152645,0.133287,0.036580,0.018253,0.016312,0.021910,0.031366,0.034866,0.053839,0.103820,0.132614,0.062758,0.025427,0.017580,0.011319,0.013994,0.007721
154,2023 to 2024,Business,0.000414,0.000152,0.000609,0.000819,0.002210,0.010750,0.045892,0.086967,0.112680,0.074659,0.060100,0.075560,0.065529,0.066300,0.063431,0.071627,0.091234,0.074635,0.039917,0.019221,0.015184,0.011300,0.006989,0.003822
155,2023 to 2024,Education,0.000000,0.000000,0.000000,0.000306,0.000044,0.000112,0.002540,0.097998,0.377559,0.017586,0.009686,0.011606,0.014716,0.008445,0.027963,0.332816,0.054838,0.026740,0.007727,0.004711,0.002080,0.001865,0.000498,0.000166
156,2023 to 2024,Escort education,0.000000,0.000000,0.000071,0.000129,0.000049,0.000220,0.001812,0.051490,0.387148,0.034087,0.003753,0.012108,0.010911,0.005194,0.093015,0.309636,0.052116,0.028840,0.005281,0.002605,0.000963,0.000209,0.000304,0.000058
157,2023 to 2024,Shopping,0.000166,0.000024,0.000000,0.000000,0.000011,0.000479,0.003359,0.012883,0.031816,0.072451,0.117920,0.124450,0.111161,0.093734,0.087138,0.075723,0.072696,0.064742,0.057175,0.042038,0.021337,0.007617,0.002203,0.000876
158,2023 to 2024,"Other work, other escort and personal business",0.000689,0.000234,0.000224,0.000071,0.000904,0.002833,0.009161,0.043429,0.093261,0.074097,0.076766,0.074615,0.081419,0.070155,0.067627,0.083308,0.090202,0.085803,0.059412,0.039304,0.023725,0.011530,0.007551,0.003681
159,2023 to 2024,"Visiting friends, entertainment and sport",0.001163,0.000473,0.000311,0.000318,0.000465,0.002845,0.007382,0.015598,0.027901,0.047855,0.057078,0.063067,0.069110,0.062209,0.060700,0.072341,0.075998,0.093584,0.108144,0.087925,0.055248,0.044755,0.029340,0.016191
160,2023 to 2024,"Holiday, day trip and other",0.000273,0.000117,0.000240,0.000665,0.001823,0.006942,0.021378,0.047803,0.063796,0.077384,0.089399,0.083460,0.070076,0.071872,0.070719,0.077639,0.074445,0.070837,0.064082,0.055170,0.027522,0.012628,0.008782,0.002949
161,2023 to 2024,All purposes,0.000766,0.000373,0.000318,0.000547,0.001795,0.006920,0.019929,0.057365,0.124588,0.055400,0.060016,0.062051,0.061212,0.055794,0.063343,0.116113,0.078482,0.078493,0.059203,0.041706,0.024498,0.015198,0.010498,0.005391


## 7. Validation

Validate the processed dataset before exporting.

In [11]:
hourly_columns = temporal_probability_matrix.columns[2:]

row_sum = temporal_probability_matrix[hourly_columns].sum(axis=1)

row_sum

153    1.0
154    1.0
155    1.0
156    1.0
157    1.0
158    1.0
159    1.0
160    1.0
161    1.0
dtype: float64

In [12]:
np.allclose(row_sum, 1.0)

True

## 8. Export

Export the processed dataset for later integration.

In [13]:
temporal_probability_matrix['Trip purpose']

153                                         Commuting
154                                          Business
155                                         Education
156                                  Escort education
157                                          Shopping
158    Other work, other escort and personal business
159         Visiting friends, entertainment and sport
160                       Holiday, day trip and other
161                                      All purposes
Name: Trip purpose, dtype: object

In [14]:
temporal_probability_matrix["Trip purpose"] = (
    temporal_probability_matrix["Trip purpose"].replace({

        "Escort education": "Escort Education",

        "Other work, other escort and personal business":
            "Other Work, Other Escort and Personal Business",

        "Visiting friends, entertainment and sport":
            "Visiting Friends, Entertainment and Sport",

        "Holiday, day trip and other":
            "Holiday, Day Trip and Other"

    })
)

In [15]:
temporal_probability_matrix = temporal_probability_matrix[
    temporal_probability_matrix["Trip purpose"] != "All purposes"
]

In [16]:
temporal_probability_matrix.to_json(
    output_data_path / output_file,
    orient="records",
    indent=4
)

## 9. Summary

### Input

- National Travel Survey (NTS)

### Output

- Standardised behavioural JSON

### Processing Completed

- Dataset cleaned
- Behaviour processed
- Validation passed
- JSON exported